# MIND-small: Knowledge-graph embeddings and history-candidate matching


This notebook engineers two model-ready inputs: (1) a 100-dimensional article entity vector and (2) a cosine similarity score for each user–candidate pair. It keeps click labels as targets for later evaluation and not ingredients of the engineered similarity. Impressions are split by date before candidate expansion.

## Relevant Features from 5.1

The following raw features from Section 5.1 of the Interim Report are used to construct the knowledge-graph features.

| Feature             | One row per        | Definition                                                                                                        |
| ------------------- | ------------------ | ----------------------------------------------------------------------------------------------------------------- |
| `news_id`           | `news_id`          | Unique identifier of a news article, used to link article metadata with user histories and candidate impressions. |
| `title_entities`    | `news_id`          | Wikidata entities identified in the article title, used to retrieve pretrained entity embeddings.                 |
| `abstract_entities` | `news_id`          | Wikidata entities identified in the article abstract, used to retrieve pretrained entity embeddings.              |
| Entity embedding    | Wikidata entity ID | Pretrained 100-dimensional vector representing a Wikidata entity, retrieved from `entity_embedding.vec`.          |
| `impression_id`     | `event_key`        | Identifier of a recommendation event, combined with its source file to uniquely identify an impression.           |
| `user_id`           | `event_key`        | Anonymised identifier of the user associated with the impression.                                                 |
| `history`           | `event_key`        | Sequence of previously clicked news articles used to construct the historical user entity profile.                |
| `impressions`       | `event_key`        | Candidate news articles and their associated click labels, used to calculate candidate-level entity similarity.   |

## Engineered Feature Summary

The following features are constructed from the raw data to provide additional information for the recommendation model.

| Feature | One row per | Definition |
|---|---|---|
| `article_entity_vector` (`entity_0`–`entity_99`) | `news_id` | 100-dimensional vector obtained by averaging the pretrained embeddings of distinct entities found in an article's title and abstract. |
| `user_entity_profile` | `event_key` | 100-dimensional vector obtained by averaging the entity vectors of articles in the user's supplied click history. |
| `entity_similarity` | `event_key`, `candidate_position` | Cosine similarity between the user's historical entity profile and the candidate article's entity vector. |
| `label` | `event_key`, `candidate_position` | Binary click outcome indicating whether the candidate article was clicked. Used as the prediction target rather than an input feature. |

Articles without usable entity embeddings are assigned zero vectors. Similarly, users without usable historical entity information receive zero profiles, resulting in a similarity score of zero.

The user entity profile is calculated internally during feature engineering, while the article entity vectors and user–candidate similarity scores are exported for subsequent model integration.


## Step 1 — Setup

In [1]:
from pathlib import Path
import os
import csv
import json
import numpy as np
import pandas as pd

def find_project_root():
    configured = os.environ.get('BT4222_PROJECT_ROOT')
    candidates = ([Path(configured).expanduser()] if configured else []) + [Path.cwd(), *Path.cwd().parents]
    if Path('/content').exists():
        candidates += [Path('/content/bt4222-main'), Path('/content/bt4222')]
    for candidate in candidates:
        candidate = candidate.resolve()
        if (candidate / 'src' / 'data_loading.py').is_file():
            return candidate
    raise FileNotFoundError('Repository not found. Run inside bt4222-main or set BT4222_PROJECT_ROOT.')

PROJECT_ROOT = find_project_root()
os.environ['BT4222_PROJECT_ROOT'] = str(PROJECT_ROOT)
RAW_DIR = Path(os.environ.get('MIND_RAW_DIR', str(PROJECT_ROOT / 'data/raw'))).expanduser().resolve()
TRAIN_DIR = RAW_DIR / 'MINDsmall_train'
DEV_DIR = RAW_DIR / 'MINDsmall_dev'
FEATURE_DIR = PROJECT_ROOT / 'outputs' / 'features' / 'entity_embeddings_v2_split_first'

# Verification
required = [TRAIN_DIR/'news.tsv', TRAIN_DIR/'behaviors.tsv', TRAIN_DIR/'entity_embedding.vec',
            DEV_DIR/'news.tsv', DEV_DIR/'behaviors.tsv']
for path in required:
    print(('OK' if path.is_file() else 'MISSING'), path)
assert all(p.is_file() for p in required), 'One or more required raw files are missing.'
if FEATURE_DIR == RAW_DIR or RAW_DIR in FEATURE_DIR.parents:
    raise ValueError('Feature output must not be inside the raw data folder.')
FEATURE_DIR.mkdir(parents=True, exist_ok=True)

print('Repository:', PROJECT_ROOT)
print('Raw data:', RAW_DIR)
print('Output folder:', FEATURE_DIR)

FileNotFoundError: Repository not found. Run inside bt4222-main or set BT4222_PROJECT_ROOT.

## Step 2 — Chronological impression split before feature construction

### Data Preparation and Temporal Splitting

To maintain consistency with Section 3.4 of the Interim Report, impression events are divided chronologically before candidate expansion and feature generation.

| Partition | Period | Purpose |
|---|---|---|
| Training | 9–12 November 2019 | Model training |
| Validation | 13–14 November 2019 | Model selection and hyperparameter tuning |
| Test | 15 November 2019 | Final evaluation |

In [ ]:
behavior_columns = ['impression_id', 'user_id', 'time', 'history', 'impressions']
def load_events(path):
    df = pd.read_csv(path, sep='\t', header=None, names=behavior_columns, dtype=str)
    df['time'] = pd.to_datetime(df['time'], errors='raise')
    assert df['impression_id'].notna().all()
    assert not df['impression_id'].duplicated().any(), f'Duplicate impression IDs in {path}'
    return df

original_train_events = load_events(TRAIN_DIR / 'behaviors.tsv')
test_events = load_events(DEV_DIR / 'behaviors.tsv')

train_events = original_train_events.loc[
    (original_train_events['time'] >= pd.Timestamp('2019-11-09')) &
    (original_train_events['time'] < pd.Timestamp('2019-11-13'))
].copy()
val_events = original_train_events.loc[
    (original_train_events['time'] >= pd.Timestamp('2019-11-13')) &
    (original_train_events['time'] < pd.Timestamp('2019-11-15'))
].copy()

# Verification
assert len(train_events) + len(val_events) == len(original_train_events)
assert (test_events['time'].dt.date == pd.Timestamp('2019-11-15').date()).all()
assert set(train_events['impression_id']).isdisjoint(set(val_events['impression_id']))
assert train_events['time'].max() < val_events['time'].min() < test_events['time'].min()

for name, events in [('train', train_events), ('val', val_events), ('test', test_events)]:
    print(name, 'whole impression events:', len(events), 'dates:', events['time'].min(), 'to', events['time'].max())

# Hardcode, from counts documented in Interim 3.4
assert (len(train_events), len(val_events), len(test_events)) == (95071, 61894, 73152)

## Step 3 — Article knowledge-graph embeddings

We then read Wikidata IDs from each article's title and abstract, find their pretrained embeddings in entity_embedding.vec, and take the unweighted average of distinct matching IDs. This turns each article into a fixed-length numerical representation. Articles without a usable embedding receive a zero vector.

In [ ]:
entity_embeddings = {}
with open(TRAIN_DIR / 'entity_embedding.vec', encoding='utf-8') as f:
    for line in f:
        parts = line.split()
        if not parts:
            continue
        entity_embeddings[parts[0]] = np.asarray(parts[1:], dtype=np.float32)
assert entity_embeddings, 'No pretrained entity embeddings found'
embedding_dim = len(next(iter(entity_embeddings.values())))
assert all(len(v) == embedding_dim for v in entity_embeddings.values())
print('Pretrained entities:', len(entity_embeddings), 'dimensions:', embedding_dim)

news_columns = ['news_id','category','subcategory','title','abstract','url','title_entities','abstract_entities']
news_frames = [pd.read_csv(folder/'news.tsv', sep='\t', header=None, names=news_columns, dtype=str)
               for folder in (TRAIN_DIR, DEV_DIR)]
news = pd.concat(news_frames, ignore_index=True).drop_duplicates('news_id', keep='first')

def entity_ids(value):
    if pd.isna(value) or not value.strip():
        return []
    return [item['WikidataId'] for item in json.loads(value) if item.get('WikidataId')]

def make_article_vector(row):
    ids = dict.fromkeys(entity_ids(row.title_entities) + entity_ids(row.abstract_entities))
    found = [entity_embeddings[eid] for eid in ids if eid in entity_embeddings]
    return np.mean(found, axis=0).astype(np.float32) if found else np.zeros(embedding_dim, dtype=np.float32)

article_vectors = {row.news_id: make_article_vector(row) for row in news.itertuples(index=False)}
zero_articles = sum(not np.any(v) for v in article_vectors.values())
print('Unique articles:', len(article_vectors), 'zero-vector articles:', zero_articles)

## Step 4 — User-history entity profile and candidate matching

For each already-split impression, average the entity vectors of its previously clicked articles to represent the user's interests. Then expand only that partition's candidate list, and compute cosine similarity between each candidate vector and its impression's history profile.


In [ ]:
def history_profile(history):
    if pd.isna(history) or not history.strip():
        return np.zeros(embedding_dim, dtype=np.float32)
    vectors = [article_vectors[nid] for nid in history.split()
               if nid in article_vectors and np.any(article_vectors[nid])]
    return np.mean(vectors, axis=0).astype(np.float32) if vectors else np.zeros(embedding_dim, dtype=np.float32)

def export_partition(events, name):
    output = FEATURE_DIR / f'{name}_entity_similarity.csv'
    candidate_count = 0
    missing_candidates = 0
    empty_profiles = 0
    with output.open('w', newline='', encoding='utf-8') as f:
        writer = csv.writer(f)
        writer.writerow(['impression_id','user_id','candidate_news_id','entity_similarity','label'])
        for row in events.itertuples(index=False):
            profile = history_profile(row.history)
            norm_profile = np.linalg.norm(profile)
            if norm_profile == 0:
                empty_profiles += 1
            tokens = row.impressions.split() if isinstance(row.impressions, str) else []
            for token in tokens:
                news_id, label = token.rsplit('-', 1)
                if label not in ('0', '1'):
                    raise ValueError(f'Unexpected click label: {token}')
                vector = article_vectors.get(news_id)
                if vector is None:
                    missing_candidates += 1
                    similarity = 0.0
                else:
                    denom = norm_profile * np.linalg.norm(vector)
                    similarity = float(np.dot(profile, vector) / denom) if denom > 0 else 0.0
                writer.writerow([row.impression_id, row.user_id, news_id, similarity, int(label)])
                candidate_count += 1
    print(f'{name}: {len(events):,} events -> {candidate_count:,} candidates; '
          f'{empty_profiles:,} empty/zero history profiles; {missing_candidates:,} unknown candidates')
    return candidate_count

candidate_counts = {}
for name, events in [('train', train_events), ('val', val_events), ('test', test_events)]:
    candidate_counts[name] = export_partition(events, name)

## Step 5 — Export engineered features to CSV



In [ ]:
article_df = pd.DataFrame.from_dict(article_vectors, orient='index')
article_df.columns = [f'entity_{i}' for i in range(embedding_dim)]
article_df.index.name = 'news_id'
article_df.reset_index().to_csv(FEATURE_DIR/'article_entity_vectors.csv', index=False)
print('Saved article_entity_vectors.csv:', len(article_df), 'articles')

## Step 6 — Verify the split, row counts and output

Verify that no candidate is dropped, the chronological event counts match the Interim Report, the exported files have the expected columns, and the output files exist. These are only structural checks.

In [ ]:
expected_candidates = {'train': 3389342, 'val': 2454102, 'test': 2740998}
for name, events in [('train', train_events), ('val', val_events), ('test', test_events)]:
    expected_from_events = events['impressions'].fillna('').map(lambda s: len(s.split())).sum()
    assert candidate_counts[name] == expected_from_events, f'Candidate mismatch for {name}'
    assert candidate_counts[name] == expected_candidates[name], f'Unexpected standard MINDsmall candidate count for {name}'
    file_path = FEATURE_DIR/f'{name}_entity_similarity.csv'
    sample = pd.read_csv(file_path, nrows=5, dtype={'impression_id':str, 'user_id':str})
    assert sample.columns.tolist() == ['impression_id','user_id','candidate_news_id','entity_similarity','label']
    print(name, 'verified:', candidate_counts[name], 'candidates; file size:', round(file_path.stat().st_size/1024**2, 1), 'MiB')

print('\nAll outputs:')
for path in sorted(FEATURE_DIR.glob('*.csv')):
    print(path.name, round(path.stat().st_size/1024**2, 1), 'MiB')
print('Finished: split occurred BEFORE vector generation and candidate expansion.')